**Axiomas**:
* Os usuários têm preferências latentes, que o modelo não precisa prever, mas precisa antecipar.
* Distância percorrida é proxy para interesse.
* Todo produto ofertado é adquirido.
* As preferências dos clientes não mudam.

**Dificuldades**
* Criar a fórmula que relaciona o quanto um usuário consegue se deslocar, o quanto ele prefere certo item e a distância entre o usuário e o item.
* A criação de uma matriz com loop de convergência aleatória pode i) nunca convergir ou ii) não ter a distribuição esparsa esperada.
* Se os atributos dos usuários não influenciarem as preferências ocultas, o cold start dos usuários ignorará tais preferências.

**Metas** - definidas arbitrariamente
* 100 Usuário
* 200 Produtos
* 20 marcas
* 10 categorias

Pensei em algo mais essencial:
* Há 20 produtos, o que é pouco, Pensei em expandir para 100 produtos. A tabela de produtos precisa de informações que influenciariam nas escolhas dos usuários. Já que estamos usando dados fictícios, temos licença poética (ou licença acadêmica) para resumir o que importa. Podemos ter o id do produto, a categoria e a marca.
* Como o app não vai levar em conta os dados do produto para recomendar, exceto no cold start, preço e quantidade não são relevantes.
* A distância é para os itens individuais, não para os produtos (toda padaria tem pão francês). Além disso, a distância é a variável de interesse na recomendação (o axioma da distância é: "quem se desloca mais para obter um produto é quem mais tem interesse nele).
* Para a tabela de usuários, precisaremos de um conjunto de colunas de preferências. São essas preferências que o sistema tenta prever (ou melhor, antecipar).
* Essa preferência se dá pelo produto, pela categoria e pela marca. Se tivermos 100 produtos, 20 marcas e 10 categorias, precisaremos de 130 colunas para todos os usuários.
* Essas colunas contém valores de 0 a 1, com o "nível de preferência" do usuário. Se um usuário tem preferência 0.5 por um produto, 0.3 pela categoria daquele produto e 0.9 pela marca daquele produto, o score daquele item é 1.7, ou 0.57: precisa ser normalizado porque isso vai refletir na distância que o cliente percorre atrás do produto.
* Os clientes precisam também de uma posição geográfica, com latitude e longitude. Podemos definir um território fictício que vai de $-G$ a $G$ nos eixos X e Y,  e os clientes são gerados com essa localização aleatória.
* Os clientes também possuem um coeficiente de "unidade de deslocamento", para simular a diferença de dificuldade entre clientes para adquirir produtos mais distantes. Um cliente com "unidade de deslocamento" igual a 1 se deslocaria por 1 unidade de distância com a mesma dificuldade com que um usuário com "unidade de deslocamento" 2 se deslocaria por 2 unidades de distância. Logo, quanto maior, mais "móvel" é o usuário.
* A tabela de iteração é a tabela de itens unitários, oferecidos aos clientes. Um produto aleatório (id_produto) é disponibilizado para todos os clientes a cada rodada.
* O algoritmo verifica, para cada cliente, o ranking de aquisição: o cliente mais propício a adquirir o item é registrado na tabela de aquisição.
    * A priori, se todos os clientes fossem iguais, o cliente mais próximo do item o adquire.
    * Se todos os itens partissem do mesmo ponto e estão sob as mesmas preferências, o usuário com maior unidade de deslocamento adquire o produto.
    * Se todos os itens partissem do mesmo ponto e todos os usuários tivessem o mesmo valor de unidade de deslocamento, o usuário com maior preferência pelo produto o adquire.
* Desta forma, para um usuário $u$ e um item $i$, sendo $R_{u,i}$ o ranking de aquisição, $UD_u$ a unidade de deslocamento do usuário, $Pr_{u,i}$ a preferência do usuário $u$ pelo item $i$, e $D_{u,i}$ a distância Manhattan entre o usuário e o item, temos:  
  
$R_{u,i}=\frac{D_{u,i}}{UD_u \cdot Pr_{u,i}}$  
  
* A iteração seleciona cada produto $n$ vezes, com um posicionamento aleatório no plano G a cada vez, escolhe o usuário que o adquire, e registra na tabela de itens adquiridos.

Isso não ficou nada essencial.

# Bibliotecas

In [83]:
import numpy as np
import pandas as pd
import itertools
from tqdm import tqdm

# Váriáveis de Controle

In [84]:
# Seed para reprodutibilidade
np.random.seed(8)

In [85]:
n_usuarios = 1000
n_produtos = 100
n_categorias = 10
n_marcas = 20

max_renda = 5000
min_renda = 95
max_moradores = 10

# A simulação geométrica vai de -tamanho_espaco a +tamanho_espaco
tamanho_espaco = 10000

# número de usuários elegíveis dentro do alcance do item ofertado
n_elegiveis = 200

# tamanho da tabela de ofertas
n_ofertas = 10000

# quantidade de ofertas de um mesmo produto por rodada
n_copias = 4

# múltiplos de ofertas por produto
mlt_ofr = 5

In [86]:
4//2

2

In [87]:
# Ponderação das preferências

K = 2
C = 5
M = 5
E = 1

In [88]:
# Espaços amostrais

def amostra_padrao(n):
    return np.random.uniform(0.001,0.999,n)
    
def amostra_espacial(n):
    return np.random.uniform(-10000,10000,n)

# Função de preferências por marca/categoria de acordo com as variáveis socioeconômicas numéridas dos usuários
def pref_socioeco(valor, media, sigma):
    z = (valor - media) / sigma
    pref = np.exp(-(z ** 2) / 2)
    # Os valores podem ser bem baixos, embora não cheguem a zero; por garantia, normalizarei na saída da função.
    return 0.001 + pref * 0.998

In [89]:
# Variáveis nativas

# Usuários
var_usuario = ('renda_per_capita','recebe_beneficio','qtd_moradores')

# Etapa 1 - Dados Socioeconômicos dos Usuários

In [90]:
usuarios = pd.DataFrame({
    'id_usuario': [f'U_{i}' for i in range(n_usuarios)],
    'renda_per_capita': np.round(np.random.uniform(min_renda, max_renda, n_usuarios),2),
    'qtd_moradores': np.random.randint(1, max_moradores + 1, n_usuarios),
    'recebe_beneficio': np.random.randint(0, 2, n_usuarios),
    'latitude': amostra_espacial(n_usuarios),
    'longitude': amostra_espacial(n_usuarios),
    'unidade_deslocamento': amostra_padrao(n_usuarios)
})

In [91]:
usuarios.head(5)

,id_usuario,renda_per_capita,qtd_moradores,recebe_beneficio,latitude,longitude,unidade_deslocamento
0,U_0,4379.17,8,0,2342.448630,7484.490925,0.790573
1,U_1,4845.69,6,1,-2344.604144,-1294.426639,0.055947
2,U_2,4358.40,8,1,6706.893956,-457.944710,0.445669
3,U_3,2698.85,8,1,4308.236619,-7154.827478,0.633112
4,U_4,1236.53,8,1,4642.484119,7997.681540,0.745822


In [92]:
# Variância amostral das variáveis numéricas dos usuários, necessárias para a densidade da normal

sigma_renda = np.std(usuarios['renda_per_capita'], ddof=1)
sigma_moradores = np.std(usuarios['qtd_moradores'], ddof=1)

# Etapa 2 - Categorias e Marcas

In [93]:
# Categorias

categorias = pd.DataFrame({
    'id_categoria': [f'C_{i}' for i in range(n_categorias)],
    'renda_per_capita': np.round(np.random.uniform(95, max_renda, n_categorias),2),
    'qtd_moradores': np.random.randint(1, max_moradores + 1, n_categorias),
    'recebe_beneficio_0': amostra_padrao(n_categorias),
    'recebe_beneficio_1': amostra_padrao(n_categorias)
})

In [94]:
categorias.head(20)

,id_categoria,renda_per_capita,qtd_moradores,recebe_beneficio_0,recebe_beneficio_1
0,C_0,4218.26,9,0.677314,0.546447
1,C_1,2796.39,3,0.786584,0.096140
2,C_2,4674.76,9,0.920121,0.931643
3,C_3,3324.47,10,0.613001,0.229847
4,C_4,1981.80,6,0.494723,0.164001
5,C_5,2771.25,5,0.193374,0.323643
6,C_6,2316.82,4,0.403227,0.229857
7,C_7,366.74,6,0.124319,0.200807
8,C_8,4987.57,8,0.120042,0.861363
9,C_9,1122.47,10,0.601729,0.059024


In [95]:
# Categorias

marcas = pd.DataFrame({
    'id_marca': [f'M_{i}' for i in range(n_marcas)],
    'renda_per_capita': np.round(np.random.uniform(95, max_renda, n_marcas),2),
    'qtd_moradores': np.random.randint(1, max_moradores + 1, n_marcas),
    'recebe_beneficio_0': amostra_padrao(n_marcas),
    'recebe_beneficio_1': amostra_padrao(n_marcas)
})

In [96]:
marcas.head(20)

,id_marca,renda_per_capita,qtd_moradores,recebe_beneficio_0,recebe_beneficio_1
0,M_0,4426.75,8,0.445705,0.551734
1,M_1,1959.61,2,0.569062,0.862135
2,M_2,1940.28,5,0.672910,0.432144
3,M_3,470.99,7,0.284712,0.961614
4,M_4,4972.53,3,0.452645,0.646423
5,M_5,3346.71,1,0.543700,0.229462
6,M_6,3566.39,9,0.690685,0.075444
7,M_7,4502.73,7,0.590679,0.792759
8,M_8,1666.41,2,0.803755,0.954333
9,M_9,3912.09,5,0.715900,0.947822


# Etapa 3 - Produtos

In [97]:
# Para garantir que uma marca não tenha produtos concorrentes, criei e aleatorizei as combinações de marca e categoria

combinacoes = list(itertools.product(marcas['id_marca'], categorias['id_categoria']))

# Escolha aleatória de n_produtos combinações
np.random.shuffle(combinacoes)
combinacoes = combinacoes[:n_produtos]


In [98]:
len(combinacoes)

100

In [99]:
n_produtos

100

In [100]:
# Tabela de produtos

produtos = pd.DataFrame({
    'id_produto': [f'P_{i}' for i in range(n_produtos)],
    'id_marca': [i[0] for i in combinacoes],
    'id_categoria': [i[1] for i in combinacoes]
})

In [101]:
# Coluna de número de vezes que cada produto é ofertado

np.random.seed(8)

soma_ofertas = n_ofertas // n_copias

oferta_aleatoria = np.random.choice(range(1, soma_ofertas//mlt_ofr), n_produtos - 1, replace=False)*mlt_ofr
print(oferta_aleatoria)

oferta_cortes = np.sort(oferta_aleatoria)
print(oferta_cortes)

# A variável, necessariamente, vai de 0 a 10000, e os valores sorteados anteriormente são os números onde há os cortes
# A soma das distâncias entre os cortes
oferta = np.diff(np.concatenate([[0], oferta_cortes, [soma_ofertas]]))

print(oferta)
print(sum(oferta))

produtos['oferta'] = oferta

[1855  285  750 1240  490  650   45  325 1825 1870  630 1285  760 2405
 2095 1565 1255 2075   65 1395 1040 2105    5 2345  140 1415 1450 2085
 1535  155  910 1180  255  835 1270 1045 1475 1365  815  560  450  660
 1560 1070  335 1140 2250  905 2155  830  605 2300  550  930 1500  210
 2020 2370 2455 1570 1655  635 1435 2010 2285  885 2175 1925 1375 2015
 1195  735 1125 2050  345  935  485  170 2390 1480 2240  890 1645 1890
 2030 1675  280  300  395  975 2220 1545   25  190  365  540 1300 1895
 1780]
[   5   25   45   65  140  155  170  190  210  255  280  285  300  325
  335  345  365  395  450  485  490  540  550  560  605  630  635  650
  660  735  750  760  815  830  835  885  890  905  910  930  935  975
 1040 1045 1070 1125 1140 1180 1195 1240 1255 1270 1285 1300 1365 1375
 1395 1415 1435 1450 1475 1480 1500 1535 1545 1560 1565 1570 1645 1655
 1675 1780 1825 1855 1870 1890 1895 1925 2010 2015 2020 2030 2050 2075
 2085 2095 2105 2155 2175 2220 2240 2250 2285 2300 2345 2370 2390 2405

In [102]:
produtos.head(10)

,id_produto,id_marca,id_categoria,oferta
0,P_0,M_2,C_9,5
1,P_1,M_0,C_3,20
2,P_2,M_10,C_9,20
3,P_3,M_11,C_9,20
4,P_4,M_13,C_5,75
5,P_5,M_18,C_9,15
6,P_6,M_10,C_2,15
7,P_7,M_18,C_4,20
8,P_8,M_2,C_6,20
9,P_9,M_12,C_8,45


# Etapa 4 - Geração de Preferências Latentes

In [103]:
# Função de apply
# Trata a linha como se fosse um dicionário, cujas chaves são os nomes das colunas

def calcula_pref_linha(usuario, df_ref, sigma_renda, sigma_moradores):
    pref_renda = pref_socioeco(usuario['renda_per_capita'], df_ref['renda_per_capita'], sigma_renda)
    pref_moradores = pref_socioeco(usuario['qtd_moradores'], df_ref['qtd_moradores'], sigma_renda)

    if usuario['recebe_beneficio'] == 0:
        col_beneficio = 'recebe_beneficio_0'
    else:
        col_beneficio = 'recebe_beneficio_1'
    pref_beneficio = df_ref[col_beneficio]

    return np.mean([pref_renda, pref_moradores, pref_beneficio])

In [104]:
# Função que aplica a outra função acima e gera a coluna dentro do dataframe

def adiciona_pref_referente(id_ref, df_usuarios, df_ref, sigma_renda, sigma_moradores):
    df_usuarios[id_ref] = df_usuarios.apply(
        calcula_pref_linha,
        axis = 1,
        df_ref = df_ref,
        sigma_renda = sigma_renda,
        sigma_moradores = sigma_moradores
    )

In [105]:
# Preferências por marcas

for _, marca in marcas.iterrows():
    adiciona_pref_referente(marca["id_marca"], usuarios, marca, sigma_renda, sigma_moradores)

In [106]:
# Preferências por categorias

for _, categoria in categorias.iterrows():
    adiciona_pref_referente(categoria["id_categoria"], usuarios, categoria, sigma_renda, sigma_moradores)

In [107]:
# Preferências por produtos

for _, produto in produtos.iterrows():
    usuarios[produto["id_produto"]] = amostra_padrao(n_usuarios)

C:\Users\fnsb\AppData\Local\Temp\ipykernel_43740\11827805.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  usuarios[produto["id_produto"]] = amostra_padrao(n_usuarios)
C:\Users\fnsb\AppData\Local\Temp\ipykernel_43740\11827805.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  usuarios[produto["id_produto"]] = amostra_padrao(n_usuarios)
C:\Users\fnsb\AppData\Local\Temp\ipykernel_43740\11827805.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has 

In [108]:
usuarios.head(5)

,id_usuario,renda_per_capita,qtd_moradores,recebe_beneficio,latitude,longitude,unidade_deslocamento,M_0,M_1,M_2,...,P_90,P_91,P_92,P_93,P_94,P_95,P_96,P_97,P_98,P_99
0,U_0,4379.17,8,0,2342.448630,7484.490925,0.790573,0.814376,0.597472,0.630324,...,0.290131,0.817202,0.020173,0.541860,0.652564,0.511080,0.621662,0.769920,0.140206,0.240869
1,U_1,4845.69,6,1,-2344.604144,-1294.426639,0.055947,0.835312,0.660249,0.515804,...,0.001453,0.438030,0.608335,0.189610,0.385835,0.639640,0.199889,0.694748,0.460796,0.793453
2,U_2,4358.40,8,1,6706.893956,-457.944710,0.445669,0.849514,0.697093,0.551968,...,0.591593,0.939728,0.081203,0.846326,0.793310,0.127968,0.756128,0.262695,0.764230,0.711579
3,U_3,2698.85,8,1,4308.236619,-7154.827478,0.633112,0.672289,0.909992,0.764530,...,0.319764,0.010173,0.562576,0.612689,0.056302,0.021525,0.159124,0.481042,0.827223,0.411306
4,U_4,1236.53,8,1,4642.484119,7997.681540,0.745822,0.541894,0.911745,0.770477,...,0.399240,0.949451,0.256061,0.528154,0.514081,0.181650,0.874910,0.680751,0.812374,0.110841


# Etapa 5 – Simulação de Ofertas

Será necessário iterar por 10000 ofertas e, para cada uma delas, determinar o ranking dos 1000 usuários.  
Os cálculos dos usuários podem ser vetorizados, o que ajuda na performance.

In [109]:
# Distância entre o usuário e o item ofertado

def dist_usuario_oferta(usuario, lat_oferta, lon_oferta):
    return np.abs(usuario['latitude'] - lat_oferta) + np.abs(usuario['longitude'] - lon_oferta)

In [110]:
# Preferência pelo produto

def pr_usuario_oferta(usuario, id_produto, id_marca, id_categoria):
    global K
    global C
    global M
    global E

    # O fator de erro serve para que um mesmo usuário não tenha sempre a mesma preferências exata pelo mesmo produto
    e = amostra_padrao(1)[0]

    return (K*usuario[id_produto] + C*usuario[id_marca] + M*usuario[id_categoria] + E*e)/(K + C + M + E)

In [111]:
# Criação da tabela de itens adquiridos

registros = []

with tqdm(total=produtos['oferta'].sum(), desc="Progresso Total") as pbar:
    for _, produto in produtos.iterrows():
        for round in range(produto['oferta']):
            lat_oferta = amostra_espacial(1)[0]
            lon_oferta = amostra_espacial(1)[0]

            usuarios['pref'] = usuarios.apply(pr_usuario_oferta, axis = 1,
                                                id_produto = produto['id_produto'],
                                                id_marca = produto['id_marca'],
                                                id_categoria = produto['id_categoria']
                                                )

            usuarios['dist'] = usuarios.apply(dist_usuario_oferta, axis = 1,
                                                lat_oferta = lat_oferta,
                                                lon_oferta = lon_oferta
                                                )
            
            # n usuários mais próximos
            elegíveis = usuarios.nsmallest(n_elegiveis, 'dist')

            adquirentes = elegíveis.nlargest(4, 'pref') # maiores preferências

            for _, adquirente in adquirentes.iterrows():
                registros.append({
                    'id_produto':   produto['id_produto'],
                    'id_usuario':   adquirente['id_usuario'],
                    'deslocamento': adquirente['dist']
                })

            # update da barra de progresso =) a ansiedade é grande
            pbar.update(1)

itens = pd.DataFrame(registros)

Progresso Total:   0%|          | 0/2500 [00:00<?, ?it/s]C:\Users\fnsb\AppData\Local\Temp\ipykernel_43740\2477165297.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  usuarios['pref'] = usuarios.apply(pr_usuario_oferta, axis = 1,
C:\Users\fnsb\AppData\Local\Temp\ipykernel_43740\2477165297.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  usuarios['dist'] = usuarios.apply(dist_usuario_oferta, axis = 1,
Progresso Total: 100%|██████████| 2500/2500 [02:11<00:00, 19.08it/s]


In [112]:
itens.head(10)

,id_produto,id_usuario,deslocamento
0,P_0,U_958,4760.773531
1,P_0,U_981,6289.107176
2,P_0,U_16,2816.495495
3,P_0,U_454,1602.160816
4,P_0,U_208,6332.289519
5,P_0,U_800,4581.087585
6,P_0,U_340,3158.885425
7,P_0,U_309,4413.944016
8,P_0,U_822,4258.526923
9,P_0,U_274,3420.192296


In [113]:
itens.describe()

,deslocamento
count,10000.000000
mean,4774.123588
std,1935.754500
min,91.260573
25%,3392.538962
50%,4903.725011
75%,6103.243603
max,12817.056445


In [114]:
# usuários sem itens:

usuarios_com_item = itens['id_usuario'].unique()
usuarios_sem_item = usuarios[~usuarios['id_usuario'].isin(usuarios_com_item)]['id_usuario']

print(f"Usuários sem item: {len(usuarios_sem_item)}")
print(usuarios_sem_item)

Usuários sem item: 23
29      U_29
207    U_207
235    U_235
266    U_266
317    U_317
326    U_326
373    U_373
417    U_417
439    U_439
514    U_514
517    U_517
531    U_531
537    U_537
605    U_605
659    U_659
697    U_697
759    U_759
768    U_768
836    U_836
851    U_851
858    U_858
934    U_934
996    U_996
Name: id_usuario, dtype: object


# Armazenamento

In [115]:
usuarios.to_parquet('../dados_sinteticos/usuarios.parquet', engine='pyarrow', compression='snappy')
produtos.to_parquet('../dados_sinteticos/produtos.parquet', engine='pyarrow', compression='snappy')
marcas.to_parquet('../dados_sinteticos/marcas.parquet', engine='pyarrow', compression='snappy')
categorias.to_parquet('../dados_sinteticos/categorias.parquet', engine='pyarrow', compression='snappy')
itens.to_parquet('../dados_sinteticos/itens.parquet', engine='pyarrow', compression='snappy')